In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def get_scaffold_activations(model, image, stage_layer_names):
    """
    model: your trained LipiNet model
    image: single input image, shape (1, H, W, 1)
    stage_layer_names: list of layer names for scaffold
                       injection points at enc1, enc2, enc3
    """
    # Build intermediate model that outputs scaffold signals
    scaffold_outputs = [
        model.get_layer(name).output
        for name in stage_layer_names
    ]

    intermediate_model = tf.keras.Model(
        inputs=model.input,
        outputs=scaffold_outputs
    )

    activations = intermediate_model.predict(image)
    return activations


def visualize_psi_cross_script(
    models_and_images,
    stage_layer_names_per_model,
    script_names
):
    """
    models_and_images: list of (model, sample_image) tuples
    script_names: ['Devanagari', 'Arabic', 'Kannada', 'Thai']
    """
    n_scripts = len(script_names)
    n_stages = 4  # stem + enc1 + enc2 + enc3

    fig, axes = plt.subplots(
        n_scripts, n_stages,
        figsize=(16, 4 * n_scripts)
    )

    col_titles = [
        'Scaffold (Stem)',
        'PSI — Enc Stage 1',
        'PSI — Enc Stage 2',
        'PSI — Enc Stage 3'
    ]

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=13,
                                fontweight='bold', pad=10)

    for row, (script, (model, image), layer_names) in enumerate(
        zip(script_names, models_and_images,
            stage_layer_names_per_model)
    ):
        activations = get_scaffold_activations(
            model, image, layer_names
        )

        axes[row, 0].set_ylabel(script, fontsize=12,
                                 fontweight='bold',
                                 rotation=90, labelpad=10)

        for col, act in enumerate(activations):
            # Average across channels to get spatial heatmap
            act_map = act[0]  # remove batch dim

            if len(act_map.shape) == 3:
                # Average over channel dimension
                act_map = np.mean(act_map, axis=-1)

            # L2 normalise per stage for comparability
            act_map = act_map / (
                np.linalg.norm(act_map) + 1e-8
            )

            im = axes[row, col].imshow(
                act_map,
                cmap='viridis',
                interpolation='nearest'
            )
            axes[row, col].axis('off')
            plt.colorbar(im, ax=axes[row, col],
                        fraction=0.046, pad=0.04)

    plt.suptitle(
        'Cross-Script PSI Scaffold Activations\n'
        'Rows: Scripts | Columns: Encoder Stages',
        fontsize=14, fontweight='bold', y=1.02
    )

    plt.tight_layout()
    plt.savefig(
        'psi_cross_script_visualization.pdf',
        bbox_inches='tight', dpi=300
    )
    plt.savefig(
        'psi_cross_script_visualization.png',
        bbox_inches='tight', dpi=300
    )
    plt.show()
    print("Saved to psi_cross_script_visualization.pdf/.png")

In [ ]:
# how to use it
# Example usage — plug in your actual layer names
# and trained models

script_names = ['Devanagari', 'Arabic', 'Kannada', 'Thai']

# Load your trained models
model_dev    = tf.keras.models.load_model('lipinet_devanagari.h5')
model_arabic = tf.keras.models.load_model('lipinet_arabic.h5')
model_kannada= tf.keras.models.load_model('lipinet_kannada.h5')
model_thai   = tf.keras.models.load_model('lipinet_thai.h5')

# Load one sample image per script from test set
img_dev    = load_sample_image('devanagari_sample.png', size=32)
img_arabic = load_sample_image('arabic_sample.png',    size=32)
img_kannada= load_sample_image('kannada_sample.png',   size=28)
img_thai   = load_sample_image('thai_sample.png',      size=64)

models_and_images = [
    (model_dev,    img_dev),
    (model_arabic, img_arabic),
    (model_kannada,img_kannada),
    (model_thai,   img_thai),
]

# These are your scaffold injection layer names
# Print model.summary() to find the exact names
layer_names = [
    # [stem_scaffold, enc1_injection, enc2_injection, enc3_injection]
    ['scaffold_conv_dev',  'psi_enc1_dev',
     'psi_enc2_dev',  'psi_enc3_dev'],
    ['scaffold_conv_ar',   'psi_enc1_ar',
     'psi_enc2_ar',   'psi_enc3_ar'],
    ['scaffold_conv_kn',   'psi_enc1_kn',
     'psi_enc2_kn',   'psi_enc3_kn'],
    ['scaffold_conv_thai', 'psi_enc1_thai',
     'psi_enc2_thai', 'psi_enc3_thai'],
]

visualize_psi_cross_script(
    models_and_images,
    layer_names,
    script_names
)

In [ ]:
Finding Your Actual Layer Names
Run this on each model to find the exact PSI layer names:
for layer in model_dev.layers:
    if 'scaffold' in layer.name or 'psi' in layer.name:
        print(layer.name, layer.output_shape)